# Voice2Reel — تدريب كاتب السكربت والمشاهد
اختر **Runtime → Change runtime type → T4 GPU**. شغّل الخلايا بالترتيب.

هذا تخصيص لـQwen2.5-1.5B باستخدام QLoRA، وليس تدريب نموذج من الصفر.
مجموعة v2 المولّدة: 930 تدريب و95 تحقق و95 اختبار، مع بيانات تشكيل ومصطلحات وحركة إضافية. تبقى اصطناعية مؤلفة داخل المستودع ولا تكفي لإثبات جودة إنتاجية؛ راجع `05_quality_loop.ipynb` للقياس الموثّق.
الأوزان الأساسية تُنزّل من Hugging Face؛ المخرجات تُحفظ على Drive.

بما أن المستودع خاص: حمّل ZIP من GitHub، وافتح هذا الدفتر في Colab وارفع ZIP في الخلية التالية.
لا حاجة لوضع رمز GitHub داخل الدفتر. حدود Colab المجاني وتوفر GPU تختلف بين الجلسات.

In [ ]:
from pathlib import Path
import zipfile
import os
import subprocess
import sys
from google.colab import files, drive

# Download Code -> Download ZIP from your private GitHub repository, then upload it here.
if not Path('/content/Voice2Reel/pyproject.toml').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one repository ZIP from GitHub')
    destination = Path('/content/repo_upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(info.file_size for info in archive.infolist()) > 50_000_000:
            raise ValueError('Expected source-only repository ZIP, under 50 MB')
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    projects = list(destination.rglob('pyproject.toml'))
    if len(projects) != 1:
        raise ValueError('Cannot identify repository root')
    import shutil
    shutil.copytree(projects[0].parent, '/content/Voice2Reel')
    del uploaded
os.chdir('/content/Voice2Reel')
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/Voice2Reel')
WORK.mkdir(parents=True, exist_ok=True)
print('Persistent artifacts:', WORK)


## تثبيت إصدارات محددة في بيئة معزولة

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.8'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/content/v2r-env'], check=True)
PY = '/content/v2r-env/bin/python'
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args], check=True)
def run(*args):
    subprocess.run([PY, *map(str, args)], check=True)
pip_install('torch==2.8.0', 'torchaudio==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126')
pip_install('-r', 'requirements/text.txt')
run('-c', "import torch; assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'; print(torch.cuda.get_device_name(0)); print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)")
# Isolated Python environment: no kernel restart or changes to Colab's preinstalled packages.

freeze = subprocess.check_output([sys.executable, "-m", "uv", "pip", "freeze", "--python", PY], text=True)
(WORK / "01_scriptwriter_qlora_environment.txt").write_text(freeze)


## فحص البيانات

In [ ]:
run('scripts/check_data.py')
run('scripts/build_datasets.py', '--check')
run('-m', 'voice2reel.quality_gate', '--output', WORK / 'quality_cpu.json')


## اختبار GPU صغير
ينفذ خطوتين فقط للتأكد من الذاكرة والتوافق، ثم يحفظ adapter تجريبيا. لا تستعمله كنموذج نهائي.

In [ ]:
run('-m', 'voice2reel.scriptwriter.train', '--output', '/content/text_smoke', '--max-steps', '2')

## قياس النموذج الأساسي قبل التدريب
هذه مجموعة التحقق. اترك مجموعة الاختبار للتقييم النهائي بعد تثبيت الإعدادات.

In [ ]:
run('-m', 'voice2reel.evaluate', 'script', '--input', 'datasets/scripts/validation.jsonl', '--output', WORK / 'text_baseline_validation.json')

## التدريب الكامل وحفظه على Drive
اختر RUN جديدا لكل تجربة جديدة. عند انقطاع جلسة، اكتب مسار checkpoint في RESUME وأبق RUN كما هو.
هذه نقاط حفظ كاملة للاستئناف؛ مجلد adapter النهائي وحده لا يحتوي حالة optimizer.

In [ ]:
RUN = WORK / 'script_run_01'
RESUME = ''  # Example: str(RUN / 'checkpoint-12')
command = ['-m', 'voice2reel.scriptwriter.train', '--output', RUN, '--epochs', '3']
if RESUME:
    command += ['--resume', RESUME]
run(*command)


## تجربة الفكرة المكتوبة

In [ ]:
LANGUAGE = "ar"  # Use en for English scripts (base model recommended; bundled training data is Arabic).
IDEA = 'اشرح لماذا يساعد تقسيم المشروع إلى خطوات صغيرة على إنجازه'
run('-m', 'voice2reel.scriptwriter.generate', '--adapter', RUN / 'adapter', '--idea', IDEA,
    '--duration', '20', '--language', LANGUAGE, '--output', WORK / 'script.json')
print((WORK / 'script.json').read_text())


## التقييم النهائي
قارن صحة JSON وmean_content_f1 مع النموذج الأساسي على نفس مجموعة الاختبار (95 مثالاً في v2).
صحة المخطط لا تضمن صحة المعلومات أو جاذبية السكربت؛ راجع النصوص بنفسك.

In [ ]:
for adapter, label in [(None, 'baseline'), (RUN / 'adapter', 'finetuned')]:
    command = ['-m', 'voice2reel.evaluate', 'script', '--input', 'datasets/scripts/test.jsonl',
               '--output', WORK / f'text_{label}_test.json']
    if adapter:
        command += ['--adapter', adapter]
    run(*command)


## تنزيل adapter؛ يحتفظ بمرجع النموذج الأساسي ويحتاج أوزانه عند التشغيل

In [ ]:
import shutil
archive = shutil.make_archive('/content/scriptwriter_adapter', 'zip', RUN, 'adapter')
files.download(archive)
